In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime, date

In [0]:
landing = 'workspace.google_drive'

In [0]:
landing_path = '/Volumes/ipl_database/default/ipl_landing_volume'

### Raw data ingestion

In [0]:
def ingest_raw_data(type,raw_table_name):
    data = spark.sql('''
                         show 
                         tables 
                         in workspace.google_drive
                    ''').filter(f"tableName like '{type}%'").collect()
    data_files = [row.tableName for row in data]

    try:
        if spark.catalog.tableExists(f"ipl_database.bronze.{raw_table_name}"):
            raw_df = spark.table(f"ipl_database.bronze.{raw_table_name}")
            max_updated_at = raw_df.select(max('updated_at')).collect()[0][0]
            for row in data:
                df = spark.table(f'workspace.google_drive.{row.tableName}').filter(col("_fivetran_synced") > lit(max_updated_at))
                if not df.isEmpty():
                    df = df.withColumn(
                        "updated_at",
                        current_timestamp()
                    )
                    df.createOrReplaceTempView("df")
                    # spark.sql(f"""
                    #     merge into ipl_database.bronze.{raw_table_name} t
                    #     using df b 
                    #     on t.id = b.id and t.date = b.date
                    #     when not matched then insert *
                    # """)
                    df.write.format('delta').mode("append").option("mergeSchema", "true").saveAsTable(f"ipl_database.bronze.{raw_table_name}")
                    print(f"File {row.tableName} appended to ipl_database.bronze.{raw_table_name}")
                else:
                    print("Data already updated. No new data found.")
        else:
            for row in data:
                df = spark.table(f'workspace.google_drive.{row.tableName}')
                df = df.withColumn(
                    "updated_at",
                    current_timestamp()
                )
                df.write.format('delta').mode("append").option("mergeSchema", "true").saveAsTable(f"ipl_database.bronze.{raw_table_name}")
            print("Table created.")
    except Exception as e:
        print(f"raw data ingestion failed due to {e}")

In [0]:
ingest_raw_data('batting','batsman_raw')

In [0]:
ingest_raw_data('bowling','bowling_raw')

In [0]:
ingest_raw_data('extra_runs','extra_runs_raw')

In [0]:
ingest_raw_data('potm','potm_raw')